### 딥러닝 기초
#### 8x8 숫자 이미지 분류 (sklearn digits 데이터셋 사용)

In [22]:
def mini_mnist():
    """8x8 숫자 이미지 분류 (sklearn digits 데이터셋 사용)"""
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report
    import numpy as np
    import matplotlib.pyplot as plt

    # 데이터 로드
    digits = load_digits()
    X, y = digits.data, digits.target
    X = X / 16.0 # 정규화 (theta - 16 + theta - 1)

    # 원-핫 인코딩
    y_onehot = np.eye (10) [y]

    # 훈련/테스트 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_onehot, test_size=0.2, random_state=42
    )

    print("미니 MNIST 과제")
    print(f"훈련 데이터: {X_train.shape}")
    print(f"테스트 데이터: {X_test.shape}")
    print(f"클래스 수: 10개 (0-9 숫자)")
    # TODO: 다중 클래스 분류 신경망 구현
    # 1. 입력층: 64개 (8x8 이미지)
    # 2. 은닉층: 적절한 크기
    # 3. 출력층: 10개 (0-9 숫자)
    # 4. Softmax 활성화 함수
    # 5. Cross-Entropy 손실 함수
    class DigitClassifier:
        def __init__(self, input_size=64, hidden_size=64, output_size=10, learning_rate=0.5):
            self.lr = learning_rate
            self.W1 = np.random.randn(input_size, hidden_size) * 0.01
            self.b1 = np.zeros(hidden_size)
            self.W2 = np.random.randn(hidden_size, output_size) * 0.01
            self.b2 = np.zeros(output_size)

        def softmax(self, z):
            exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))

            return exp_z / np.sum(exp_z, axis=1, keepdims=True)
        
        def cross_entropy_loss(self, y_pred, y_true):
            m = y_true.shape[0]

            return -np.sum(y_true * np.log(y_pred + 1e-9)) / m

        def forward(self, X):
            self.z1 = X @ self.W1 + self.b1
            self.a1 = np.maximum(0, self.z1)
            self.z2 = self.a1 @ self.W2 + self.b2
            self.y_pred = self.softmax(self.z2)

            return self.y_pred

        def backward(self, X, y):
            m = X.shape[0]

            delta2 = self.y_pred - y
            dW2 = (self.a1.T @ delta2) / m
            db2 = np.sum(delta2, axis=0) / m

            delta1 = (delta2 @ self.W2.T) * (self.z1 > 0)
            dW1 = (X.T @ delta1) / m
            db1 = np.sum(delta1, axis=0) / m

            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2

        def predict(self, X):
            probabilities = self.forward(X)

            return np.argmax(probabilities, axis=1)

        def accuracy(self, X, y):
            predictions = self.predict(X)
            true_labels = np.argmax(y, axis=1)
            
            return np.mean(predictions == true_labels)
            
    model = DigitClassifier()

    epochs = 1500
    for epoch in range(epochs):
        y_pred = model.forward(X_train)
        model.backward(X_train, y_train)

        if epoch % 100 == 0:
            loss = model.cross_entropy_loss(y_pred, y_train)
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    # 모델 평가
    train_acc = model.accuracy(X_train, y_train)
    test_acc = model.accuracy(X_test, y_test)

    y_test_labels = np.argmax(y_test, axis=1)
    y_pred_labels = model.predict(X_test)
        
    print(f"훈련 정확도: {train_acc:.2%}")
    print(f"테스트 정확도: {test_acc:.2%}")
    print("\n각 숫자별 예측 성능:")
    print(classification_report(y_test_labels, y_pred_labels))

mini_mnist()

미니 MNIST 과제
훈련 데이터: (1437, 64)
테스트 데이터: (360, 64)
클래스 수: 10개 (0-9 숫자)
Epoch 0, Loss: 2.3027
Epoch 100, Loss: 0.2945
Epoch 200, Loss: 0.1342
Epoch 300, Loss: 0.0923
Epoch 400, Loss: 0.0719
Epoch 500, Loss: 0.0588
Epoch 600, Loss: 0.0494
Epoch 700, Loss: 0.0421
Epoch 800, Loss: 0.0363
Epoch 900, Loss: 0.0315
Epoch 1000, Loss: 0.0277
Epoch 1100, Loss: 0.0245
Epoch 1200, Loss: 0.0218
Epoch 1300, Loss: 0.0195
Epoch 1400, Loss: 0.0176
훈련 정확도: 99.93%
테스트 정확도: 96.67%

각 숫자별 예측 성능:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98        33
           1       0.93      1.00      0.97        28
           2       0.97      0.97      0.97        33
           3       0.97      0.97      0.97        34
           4       0.98      0.98      0.98        46
           5       0.94      0.94      0.94        47
           6       0.97      0.97      0.97        35
           7       1.00      0.97      0.99        34
           8       0.97      0.97  